In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime

import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils

In [9]:
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
date,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0


In [10]:
# 交易额是volume ，这里是跟昨天比较，如果暴涨是多少倍。
volume_radio = 2
dt['volume2']=talib.SMA(dt['volume'], timeperiod= 5) # 向下移动
dt2 = dt[(dt['volume']/dt['volume2'] >= 5) & (dt['close']  > dt['preclose'])]
dt2.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST,volume2
date,,,,,,,,,,,,,,


In [11]:
len(dt2)

0

In [12]:
# 我这里要绘图
volume_radio = 3
day_before = 14
days_after = 14
_days = [3,5,10]
N1 = 1.06
N2 = 3

_folder_name = f'交易量倍率{volume_radio}往前{day_before}往后{days_after}'
if not os.path.exists(_folder_name):
    os.makedirs(_folder_name) 

lst_dt = []

for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.get_data(code)
    for j in _days:
        # 计算涨跌幅度的
        dt[f'radio{j}'] = (dt['close'].shift(-j) - dt['close']) / dt['close'] * 100
    # 这里是交易额
    dt['prevolume'] = dt['volume'].shift(1)
    
    dt2 = dt.loc[(dt['close'] > N1 *  dt['preclose']) & (dt['volume'] > N2 * dt['prevolume']) & (dt['close'] < 1.11 *  dt['preclose']) ]

    # 交易额暴涨后多少天的
    if len(dt2) >0:
        # 添加到
        lst_dt.append(dt2)        
        for index in dt2.index:
            # 然后这里要取得多少天
            _before = index - datetime.timedelta(days=day_before) # 前面几天
            _after = index + datetime.timedelta(days=days_after)  # 后边几天
            _df_plot = dt.loc[_before:_after,:]                   # 截取
            _img_file = f'{code}-{index.strftime("%Y-%m-%d")}.jpg'
            _img_path = os.path.join(_folder_name, _img_file)
            Utils.utils.save_dt_img(_df_plot, _img_path)
            


100%|████████████████████████████████████████████████████████████████████████████| 5493/5493 [2:23:23<00:00,  1.57s/it]


In [13]:
dt_all = pd.concat(lst_dt)
print(len(dt_all))
for i in _days:
    _min = dt_all[f'radio{i}'].min()
    _max = dt_all[f'radio{i}'].max()
    _mean = dt_all[f'radio{i}'].mean()
    _median =  dt_all[f'radio{i}'].median()
    _std =  dt_all[f'radio{i}'].std()
    print(f'{i},mean:{_mean:5.2f}%,median:{_median:5.2f}%,std:{_std:5.2f}%, min:{_min:5.2f}%, max:{_max:5.2f}%')

73835
3,mean: 1.23%,median:-0.37%,std: 9.68%, min:-42.42%, max:73.02%
5,mean: 1.24%,median:-0.83%,std:12.84%, min:-63.03%, max:142.97%
10,mean: 1.51%,median:-1.36%,std:18.50%, min:-63.29%, max:166.61%


# 总结
交易额暴涨意味着，平均下跌开始